In [1]:
import re
import pandas as pd

In [2]:
# import nltk
# nltk.download('stopwords')
# # once per virtual environment
# # comment out after finishing download

In [3]:
# nltk.download('wordnet')
# # can use spaCy library to reduce words to base form, wordnet is for lemmatization and synonyms
# # once per virtual environment
# # comment out after finishing download 

In [4]:
# method 1: remove negations from stop words list 
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
negations={'not','no','never','nor'}
# filter out low information word 
# also filter out the negations

stop_words=stop_words-negations
text = "The cat isn't sitting on the mat."
words = text.split()
filtered_words = [word for word in words if word.lower() not in stop_words]
print(filtered_words)







['cat', 'sitting', 'mat.']


In [5]:
# import nltk
# nltk.download('punkt_tab')

# # # only download once first time 

In [6]:
# # method 2: 
from nltk.tokenize import word_tokenize
text="The cat isn't sitting on the mat."
stop_words.discard('not')
# remove not from stop_words--> survives the filter 
tokens=word_tokenize(text)
# punctuation kept as separate items:  "can't" into ["can", "'t"]
print(tokens)




# expand_text=text.replace("isn't","is not").replace("don't","do not")
# expand contractions 
expand_text=text.replace("n't"," not")
# with a front space make sure isn't split to 'is','not' instead of 'isnot'
# since word_tokenize separate item by punctuation or space 
words=word_tokenize(expand_text)
# text become list after tokenize 
print(words)
filtered_words=[word for word in words if word.lower() not in stop_words and word.isalnum()]
# word.isalnum(): word is alphabet or number--> exclude anything else 
print(filtered_words)

['The', 'cat', 'is', "n't", 'sitting', 'on', 'the', 'mat', '.']
['The', 'cat', 'is', 'not', 'sitting', 'on', 'the', 'mat', '.']
['cat', 'not', 'sitting', 'mat']


In [7]:
print(text)

The cat isn't sitting on the mat.


In [8]:
# method 3: 
df=pd.read_csv('sentiment analysis/sentiment_analysis.csv')
df=df[['sentiment','text']]
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
# default switch plural noun to singular noun
def lemma(w):
    return lemmatizer.lemmatize(lemmatizer.lemmatize(w,pos='v'),pos='n')
# set both verb and noun, verb pass first then noun 
# POS(part of speech) tagging: slower and less reliable on lowercase and informal text 
def clean_text(text):
    if not isinstance(text,str):
    # is variable(text) made from (str) datatype
        return []
    text=text.lower()
    text=re.sub(r'http\S+|www\S+',' ',text)
    # remove URLs \S match any non-whitespace character->everything up to the next space
    # "visit http://example.com/page now", it captures http://example.com/page
    text=re.sub(r'<.*?>',' ',text)
    # remove HTML tages
    # . match anything including the space 
#     .* (greedy) matches as much as possible.
#     .*? (non-greedy) matches as little as possible.
# Greedy <.*> → matches the whole string <p>Hello</p> (from first < to last >).
# Non-greedy <.*?> → matches just <p>, then continues and matches </p> separately.
    text = re.sub(r"[`’]", "'", text) 
    text=text.replace("n't"," not")
    # order is important,the replace("n't", " not") line runs before the backtick conversion. A word like don`t will still turn into don t
    # convert both `’ to '
    # the order is important, 'spostrophe' is deleted next line
    text=re.sub(r'[^a-z\s]',' ',text)
    # ^: negate 
    # a-z:any lowercase letter
    # \s match white space
    # keep only letter & spaces
    # text=re.sub(r"n't\b", " not", text)
    # \b means a word boundary->It ensures that n't is only matched if it is at the end of a complete word.
    # don't -> do not 
    text = re.sub(r'\s+', ' ', text).strip()  
    # strip()remove extra space that sit at the begining and the end
    # re.sub(r'\s+', ' ', text) collapse repeated space 
    tokens=text.split()
    # a custom whitelist prevent no being removed which distort sentiment 
    tokens=[lemma(w) for w in tokens
            if (w not in stop_words and len(w)>2) or w in negations]
    # remove short words : am/or/pm...
    # reduce vocabulary size--> reduce overfitting on small dataset
    return ' '.join(tokens)
    # return string instead of list ['cat', 'sitting', 'mat.']<--TfidfVectorizer expects a single string


df['clean_text']=df['text'].apply(clean_text)
df.head()



,sentiment,text,clean_text
0,positive,What a great day!!! Looks like dream.,great day look like dream
1,positive,"I feel sorry, I miss you here in the sea beach",feel sorry miss sea beach
2,negative,Don't angry me,not angry
3,negative,We attend in the class just for listening teac...,attend class listen teacher read slide nonsence
4,negative,"Those who want to go, let them go",want let


In [9]:
df.tail()

,sentiment,text,clean_text
494,negative,"According to , a quarter of families under six...",accord quarter family six live poverty
495,negative,the plan to not spend money is not going well,plan not spend money not go well
496,neutral,uploading all my bamboozle pictures of facebook,upload bamboozle picture facebook
497,positive,congratulations ! you guys finish a month ear...,congratulation guy finish month early booo
498,negative,"actually, I wish I was back in Tahoe. I miss...",actually wish back tahoe miss


In [10]:
from collections import Counter
all_words1=' '.join(df['clean_text'].astype(str)).split()
# join all the rows into one big string, then split() into words
# ' '.join: last word of one row stick to the first word next row, the content in '' is the connecting content, usually space
# all_words2=[alphabet for row in df['clean_text'] for alphabet in row]
# df['clean_text'] is a column of string, without row.split() drops to alphabet level
all_words2=[word for row in df['clean_text'] for word in row.split()]
sentence_list=[word for word in df['clean_text']]
# each item is full cleaned sentence ==df['clean_text'].tolist()
# counts duplicate sentence


In [11]:
counts=Counter(all_words1)
# all_words=' '.join(df['clean_text']).split()
# counts=Counter(all_words)
print(counts.most_common(10))
print(counts.most_common()[:-11:-1])

[('not', 74), ('day', 55), ('get', 42), ('go', 40), ('love', 34), ('know', 31), ('like', 30), ('happy', 24), ('today', 23), ('really', 22)]
[('tahoe', 1), ('month', 1), ('guy', 1), ('bamboozle', 1), ('money', 1), ('poverty', 1), ('six', 1), ('quarter', 1), ('accord', 1), ('proud', 1)]


In [12]:
counts=Counter(all_words2)
# all_words=' '.join(df['clean_text']).split()
# counts=Counter(all_words)
print(counts.most_common(10))
print(counts.most_common()[:-11:-1])

[('not', 74), ('day', 55), ('get', 42), ('go', 40), ('love', 34), ('know', 31), ('like', 30), ('happy', 24), ('today', 23), ('really', 22)]
[('tahoe', 1), ('month', 1), ('guy', 1), ('bamboozle', 1), ('money', 1), ('poverty', 1), ('six', 1), ('quarter', 1), ('accord', 1), ('proud', 1)]


In [13]:
counts=Counter(sentence_list)
# all_words=' '.join(df['clean_text']).split()
# counts=Counter(all_words)
print(counts.most_common(10))
print(counts.most_common()[:-11:-1])


[('grab coffee make mom breakfast', 4), ('', 3), ('sorry', 3), ('haha know cant handle fame thank', 3), ('think go fun tonight maybe change come', 3), ('thank', 3), ('great day look like dream', 2), ('feel sorry miss sea beach', 2), ('not angry', 2), ('attend class listen teacher read slide nonsence', 2)]
[('actually wish back tahoe miss', 1), ('congratulation guy finish month early booo', 1), ('upload bamboozle picture facebook', 1), ('plan not spend money not go well', 1), ('accord quarter family six live poverty', 1), ('home alone', 1), ('sorry try keep', 1), ('no not sad make proud', 1), ('go stick office without window', 1), ('thats another sponsor', 1)]


In [14]:
# all_words3 shows ('', 3). Three rows became empty after cleaning, probably because they contained only stopwords or short words.
df=df[df['clean_text']!='']
sentence_list=[word for word in df['clean_text']]
counts=Counter(sentence_list)
# all_words=' '.join(df['clean_text']).split()
# counts=Counter(all_words)
print(counts.most_common(10))
print(counts.most_common()[:-6:-1])

[('grab coffee make mom breakfast', 4), ('sorry', 3), ('haha know cant handle fame thank', 3), ('think go fun tonight maybe change come', 3), ('thank', 3), ('great day look like dream', 2), ('feel sorry miss sea beach', 2), ('not angry', 2), ('attend class listen teacher read slide nonsence', 2), ('want let', 2)]
[('actually wish back tahoe miss', 1), ('congratulation guy finish month early booo', 1), ('upload bamboozle picture facebook', 1), ('plan not spend money not go well', 1), ('accord quarter family six live poverty', 1)]
